In [1]:
import pandas as pd
import threading
import time
import os

# Reconstrução das vistas

In [ ]:
finaliza_laco = False
estado_stream = {}
lock_arquivo = threading.Lock()

metricas = ["media","contagem","maximo","minimo","mediana","desvio_padrao","variancia","moda"]

# Atualiza a vista rápida, calculando as métricas
def atualizar_vista_rapida(linha, header, var):
    global estado_stream
    
    # Seta o path da contagem incremental
    caminho = "vistas_tempo_real/contagem_incremental.csv"
    valores = [v.strip().replace('"', '') for v in linha.split(",")]

    try:
        # Pega o index da linha onde se encontra a coluna definida
        idx = header.index(var)
        valor = float(valores[idx])
        timestamp_completo = valores[0]
        data_dia = timestamp_completo.split(" ")[0]
    except:
        return

    # Verifica se tem a linha, caso nao tenha, ele cria
    chave = (data_dia, var)
    if chave not in estado_stream:
        estado_stream[chave] = {"count": 0, "soma": 0, "max": valor, "min": valor, "valores": []}

    # Guarda o estado
    estado = estado_stream[chave]
    estado["count"] += 1
    estado["soma"] += valor
    estado["max"] = max(estado["max"], valor)
    estado["min"] = min(estado["min"], valor)
    estado["valores"].append(valor)

    # Cálculo das métricas
    resultados = {
        "media": estado["soma"] / estado["count"],
        "contagem": estado["count"],
        "maximo": estado["max"],
        "minimo": estado["min"],
        "mediana": pd.Series(estado["valores"]).median(),
        "desvio_padrao": pd.Series(estado["valores"]).std(),
        "variancia": pd.Series(estado["valores"]).var(),
        "moda": pd.Series(estado["valores"]).mode()[0] if not pd.Series(estado["valores"]).mode().empty else None
    }
    
    # Trava a thread para que nenhuma outra thread altere/acesse simultaneamente.
    with lock_arquivo:
        nova_linha = {"data": timestamp_completo, "var": var, **resultados}
        df_nova = pd.DataFrame([nova_linha])
        header_necessario = not os.path.exists(caminho)
        df_nova.to_csv(caminho, mode='a', index=False, header=header_necessario)

# Salva a ultima linha da contagem incremental na vista
def reconstruir_vistas_lote():
    
    print("Reconstruindo vistas de lote (Mesclando linhas)...")
    
    # Definindo os paths de entrada e saida
    entrada = "vistas_tempo_real/contagem_incremental.csv"
    saida = "vistas_lote/vistas_de_lote copy.csv"
    
    if not os.path.exists(entrada): return
    df_real = pd.read_csv(entrada)
    if df_real.empty: return

    # Consolidar o tempo real: pega a última atualização de cada dia/variável
    df_real['data_dia'] = df_real['data'].str.split(" ").str[0]
    df_novos_dados = df_real.sort_values('data').groupby(['data_dia', 'var']).last().reset_index()
    df_novos_dados['data'] = df_novos_dados['data_dia']
    df_novos_dados = df_novos_dados.drop(columns=['data_dia'])

    if not os.path.exists(saida):
        df_novos_dados.to_csv(saida, index=False)
    else:
        df_lote = pd.read_csv(saida)
        
        # Garante que a coluna data seja string para comparação
        df_lote['data'] = df_lote['data'].astype(str)
        df_novos_dados['data'] = df_novos_dados['data'].astype(str)

        # Define o index para fazer o update na linha correta
        df_lote = df_lote.set_index(['data', 'var'])
        df_novos_dados = df_novos_dados.set_index(['data', 'var'])
        df_lote.update(df_novos_dados)
        
        # Se houver combinações (data, var) novas que não existiam no lote, adicionamos elas
        df_final = pd.concat([df_lote, df_novos_dados[~df_novos_dados.index.isin(df_lote.index)]])
        
        df_final.reset_index().to_csv(saida, index=False)
    print("Lote mesclado com sucesso!")

def stream_dados(arq):
    global finaliza_laco
    while not finaliza_laco:
        linha = arq.readline().strip()
        if not linha:
            time.sleep(0.1)
            continue
        yield linha

def monitora_linhas(arquivo):
    while not os.path.exists(arquivo) or os.path.getsize(arquivo) == 0:
        time.sleep(0.1)
    # Adiciona uma linha no arquivo de contragem incremental com as métricas calculadas para a var definida
    with open(arquivo, "r") as arq:
        header = [h.strip().replace('"', '') for h in arq.readline().strip().split(",")]
        for linha in stream_dados(arq):
            atualizar_vista_rapida(linha, header, "duration_(secs)")

def simular_stream_csv(entrada, saida, delay=0.05):
    global finaliza_laco
    
    os.makedirs(os.path.dirname(saida), exist_ok=True)
    
    with open(entrada, "r") as arq_in:
        with open(saida, "w") as arq_out:
            header = arq_in.readline()
            arq_out.write(header)
            arq_out.flush()
            for linha in arq_in:
                if finaliza_laco: break
                arq_out.write(linha)
                arq_out.flush()
                time.sleep(delay)

if __name__ == "__main__":
    
    # Seta os pahts de entrada, saida e stream
    arquivo_entrada = "dados_brutos/2017-03-21.csv"
    arquivo_stream = "dados_novos/fluxo.log"
    csv_tempo_real = "vistas_tempo_real/contagem_incremental.csv"

    os.makedirs("dados_novos", exist_ok=True)
    os.makedirs("vistas_tempo_real", exist_ok=True)

    if os.path.exists(csv_tempo_real): os.remove(csv_tempo_real)
    open(arquivo_stream, "w").close()

    # Define as threads
    t1 = threading.Thread(target=simular_stream_csv, args=(arquivo_entrada, arquivo_stream), daemon=True)
    t2 = threading.Thread(target=monitora_linhas, args=(arquivo_stream,), daemon=True)

    # Starta as threads
    t1.start()
    t2.start()

    try:
        while t1.is_alive():
            time.sleep(1)
    except KeyboardInterrupt:
        pass
    finally:
        # Finaliza as threads e salva a ultima linha da contagem incremental na vista de lote
        finaliza_laco = True
        t1.join()
        t2.join()
        reconstruir_vistas_lote()
        print("Processo finalizado.")

Reconstruindo vistas de lote (Mesclando linhas)...
Lote mesclado com sucesso!
Processo finalizado.
